In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.interpolate import CubicSpline
from scipy.interpolate import PchipInterpolator

In [ ]:
#I will be using tidal data from NOAA to model San Diego tides on November 6th, 2025
#This was one of the most recent 'King Tide' events in San Diego

In [ ]:
#define times in minutes of high and low tides
times = [141,519,950,1330]
#define tide levels in feet at high and low tides
levels = [1.68,7.51,-1.29,4.34]

In [ ]:
total_time = 1440 #total minutes in 24 hours
t_start = 0 #12:00 AM
t_end = 1439 #11:59 PM

In [ ]:
#because I could not find exact tide levels for each minute in a day, I will be using the high and low tide levels as anchors
#I will approximate the starting and ending levels, and use a cubic spline interpolation model to plot the other levels
#I used PchipInterpolator to ensure that the given tidal levels are the maximum and minimum points on the graph

In [ ]:
h_start = levels[0] + (levels[1] - levels[0])*(times[0] - t_start)/(times[1] - times[0])
print(h_start)

In [ ]:
h_end = levels[-1] + (levels[-2] - levels[-1])*(t_end - times[-1])/(times[-1] - times[-2])
h_end = np.clip(h_end, -0.5, 5.88)
print(h_end)

In [ ]:
t_points = np.concatenate(([0], times, [total_time]))
h_points = np.concatenate(([h_start], levels, [h_end]))
curve = PchipInterpolator(t_points, h_points) #create a smooth curve based on the given time and level points

In [ ]:
all_times = np.arange(0,total_time)
tide_heights_all = curve(all_times) #calculate tide levels for each minuted based on the cubic spline interpolation model created above

In [ ]:
#set up animation
fig, ax = plt.subplots(figsize = (10,6), dpi = 100)
total_frames = len(all_times)
#limits
ax.set_xlim(0, total_time)
ax.set_ylim(-2,8)
#labels
ax.set_xlabel('Time (PST)', fontsize = 12)
ax.set_ylabel('Tide Height (feet)')
fig.suptitle('San Diego King Tide Animation (November 6th, 2025)', fontsize = 16, weight = 'bold')
ax.grid(True, linestyle = '--')
#create time labels on the x axis
tick_interval = 3*60 #ticks every three hours
ax.set_xticks(np.arange(0, total_time + tick_interval, tick_interval))
#I want the minutes to show up in hh:mm AM/PM format
def time_format(x, pos):
    hours = int(x//60)
    minutes = int(x%60)
    if hours == 24: #this converts 1400 minutes to 11:59 pm
        hours = 23
        minutes = 59
    dt = pd.to_datetime('2025-11-06 00:00:00') + pd.Timedelta(minutes=hours*60 + minutes)
    return dt.strftime('%I:%M %p')
ax.xaxis.set_major_formatter(plt.FuncFormatter(time_format))
ax.plot(all_times, tide_heights_all, color = 'skyblue', alpha = 0.5, linestyle = '--', label = 'Approximate Tidal Curve')
line, = ax.plot([], [], color = 'blue', linewidth = 3, label = 'Current Tide Level')
time_marker, = ax.plot([], [], 'k--', alpha = 0.7)
water_level_dot, = ax.plot([], [], 'o', color='red', markersize=8, label = 'Current water Level')
annotation = ax.annotate(
    '', xy=(0, tide_heights_all[0]), xytext=(30,0), textcoords = 'offset points',
    arrowprops=dict(arrowstyle = "->", connectionstyle = "arc3,rad=.2", color='black'),
    bbox=dict(boxstyle="round,pad=0.5", fc = 'white', alpha = 0.7)
)
annotation.set_visible(False)
#High and Low tide markers to make the high and low tides very clean
ax.plot(times, levels, 'x', color = 'green', markersize = 10, markeredgewidth = 2, label = 'Observed Tides')

#Creating the actual animation
def animate(i):
    x_data = all_times[:i+1]
    y_data = tide_heights_all[:i+1]
    line.set_data(x_data, y_data)
    current_time = all_times[i]
    current_height = tide_heights_all[i]
    time_marker.set_data([current_time, current_time], [ax.get_ylim()[0], current_height])
    water_level_dot.set_data([current_time], [current_height])
    dt = pd.to_datetime('2025-11-06 00:00:00') + pd.Timedelta(minutes=current_time)
    time_str = dt.strftime('%I:%M %p')
    height_str = f'{current_height:.2f} ft'
    annotation.xy = (current_time, current_height)
    annotation.set_text(f'Time: {time_str}\nHeight: {height_str}')
    annotation.set_visible(True)
    plt.legend(loc='upper left')
    return line, time_marker, water_level_dot, annotation
ani = animation.FuncAnimation(fig, animate, frames = total_frames, interval = 50, blit = False, repeat = False)
ani.save('../../Downloads/tide_animation_1.mp4', writer = 'ffmpeg', fps = 60)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#I will now make an animation of normal tidal levels in San Diego
times_2 = [210,460,814,1275]
levels_2 = [4.06,3.35,5.14,0.24]
#approximation formula changes a bit
h_start_2 = levels_2[0] - (levels_2[0] - levels_2[1])*2*(times_2[0] - t_start)/(times_2[1] - times_2[0])
print(h_start_2)

In [ ]:
h_end_2 = levels_2[-1] + (levels_2[-2] - levels_2[-1])*(0.18)*(t_end - times_2[-1])/(times_2[-1] - times_2[-2])
print(h_end_2)

In [ ]:
t_points_2 = np.concatenate(([0], times_2, [total_time]))
h_points_2 = np.concatenate(([h_start_2], levels_2, [h_end_2]))
curve_2 = PchipInterpolator(t_points_2, h_points_2)
tide_heights_all_2 = curve_2(all_times)

In [ ]:
#set up animation
fig_2, ax = plt.subplots(figsize = (10,6), dpi = 100)
total_frames = len(all_times)
#limits
ax.set_xlim(0, total_time)
ax.set_ylim(-2,8)
#labels
ax.set_xlabel('Time (PST)', fontsize = 12)
ax.set_ylabel('Tide Height (feet)')
fig_2.suptitle('San Diego Normal Tide Animation (November 11th, 2025)', fontsize = 16, weight = 'bold')
ax.grid(True, linestyle = '--')
#create time labels on the x axis
tick_interval = 3*60 #ticks every three hours
ax.set_xticks(np.arange(0, total_time + tick_interval, tick_interval))
#I want the minutes to show up in hh:mm AM/PM format
def time_format(x, pos):
    hours = int(x//60)
    minutes = int(x%60)
    if hours == 24: #this converts 1400 minutes to 11:59 pm
        hours = 23
        minutes = 59
    dt = pd.to_datetime('2025-11-11 00:00:00') + pd.Timedelta(minutes=hours*60 + minutes)
    return dt.strftime('%I:%M %p')
ax.xaxis.set_major_formatter(plt.FuncFormatter(time_format))
ax.plot(all_times, tide_heights_all_2, color = 'skyblue', alpha = 0.5, linestyle = '--', label = 'Approximate Tidal Curve')
line, = ax.plot([], [], color = 'blue', linewidth = 3, label = 'Current Tide Level')
time_marker, = ax.plot([], [], 'k--', alpha = 0.7)
water_level_dot, = ax.plot([], [], 'o', color='red', markersize=8, label = 'Current water Level')
annotation = ax.annotate(
    '', xy=(0, tide_heights_all[0]), xytext=(30,0), textcoords = 'offset points',
    arrowprops=dict(arrowstyle = "->", connectionstyle = "arc3,rad=.2", color='black'),
    bbox=dict(boxstyle="round,pad=0.5", fc = 'white', alpha = 0.7)
)
annotation.set_visible(False)
#High and Low tide markers to make the high and low tides very clean
ax.plot(times_2, levels_2, 'x', color = 'green', markersize = 10, markeredgewidth = 2, label = 'Observed Tides')

#Creating the actual animation
def animate_2(i):
    x_data_2 = all_times[:i+1]
    y_data_2 = tide_heights_all_2[:i+1]
    line.set_data(x_data_2, y_data_2)
    current_time = all_times[i]
    current_height_2 = tide_heights_all_2[i]
    time_marker.set_data([current_time, current_time], [ax.get_ylim()[0], current_height_2])
    water_level_dot.set_data([current_time], [current_height_2])
    dt = pd.to_datetime('2025-11-11 00:00:00') + pd.Timedelta(minutes=current_time)
    time_str = dt.strftime('%I:%M %p')
    height_str_2 = f'{current_height_2:.2f} ft'
    annotation.xy = (current_time, current_height_2)
    annotation.set_text(f'Time: {time_str}\nHeight: {height_str_2}')
    annotation.set_visible(True)
    plt.legend(loc='upper left')
    return line, time_marker, water_level_dot, annotation
ani = animation.FuncAnimation(fig_2, animate_2, frames = total_frames, interval = 50, blit = False, repeat = False)
ani.save('../../Downloads/tide_animation_2.mp4', writer = 'ffmpeg', fps = 60)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#I will now create an animation that shows both sets of tidal data in one graph
fig_3, ax = plt.subplots(figsize = (10,6), dpi = 100)
ax.set_xlim(0,total_time)
ax.set_ylim(-2,8)
fig_3.suptitle('King Tide vs. Normal Tide', fontsize = 16, weight = 'bold')
ax.set_xlabel('Time (PST)', fontsize = 12)
ax.set_ylabel('Tide Height (feet)', fontsize = 12)
ax.grid(True, linestyle = '--', alpha = 0.6)
tick_interval = 3*60 #ticks every three hours
ax.set_xticks(np.arange(0, total_time + tick_interval, tick_interval))
#I want the minutes to show up in hh:mm AM/PM format
def time_format(x, pos):
    hours = int(x//60)
    minutes = int(x%60)
    if hours == 24: #this converts 1400 minutes to 11:59 pm
        hours = 23
        minutes = 59
    dt = pd.to_datetime('2025-11-07 00:00:00') + pd.Timedelta(minutes=hours*60 + minutes)
    return dt.strftime('%I:%M %p')
ax.xaxis.set_major_formatter(plt.FuncFormatter(time_format))

line_king, = ax.plot([], [], color = 'red', linewidth = 3, label = 'King Tide')
ax.plot(all_times, tide_heights_all, color = 'red', linestyle = '--', alpha = 0.3)
line_normal, = ax.plot([], [], color = 'blue', linewidth = 3, label = 'Normal Tide')
ax.plot(all_times, tide_heights_all_2, color = 'blue', linestyle = '--', alpha = 0.3)
time_marker, = ax.plot([], [], 'k--', alpha = 0.7)
water_level_dot, = ax.plot([], [], 'o', color = 'red', markersize = 8)
water_level_dot_2, = ax.plot([], [], 'o', color = 'blue', markersize = 8)
annotation = ax.annotate(
    '', (0, tide_heights_all[0]), xytext=(30,0), textcoords = 'offset points',
    arrowprops = dict(arrowstyle = "->", connectionstyle = "arc3,rad=.2", color = 'black'),
    bbox = dict(boxstyle = "round,pad=0.5", fc = 'white', alpha = 0.7)
)
annotation.set_visible(False)
ax.plot(times_2, levels_2, 'x', color = 'green', markersize = 10, markeredgewidth = 2, label = 'Observed Normal Tides')
ax.plot(times, levels, 'x', color = 'purple', markersize = 10, markeredgewidth = 2, label = 'Observed King Tides')

def animate_both(i):
    x_data_3 = all_times[:i+1]
    y_data_king = tide_heights_all[:i+1]
    line_king.set_data(x_data_3, y_data_king)
    y_data_normal = tide_heights_all_2[:i+1]
    line_normal.set_data(x_data_3, y_data_normal)
    current_time = all_times[i]
    current_height_king = tide_heights_all[i]
    current_height_normal = tide_heights_all_2[i]
    time_marker.set_data([current_time, current_time], [ax.get_ylim()[0], max(current_height_king, current_height_normal)])
    water_level_dot.set_data([current_time], [current_height_king])
    water_level_dot_2.set_data([current_time], [current_height_normal])
    dt = pd.to_datetime('2025-11-07 00:00:00') + pd.Timedelta(minutes = current_time)
    time_str = dt.strftime('%I:%M %p')
    text_content = (f'Time: {time_str}\n' f'King Tide: {current_height_king:.2f} ft\n' f'Normal Tide: {current_height_normal:.2f} ft')
    annotation.xy = (current_time, current_height_king)
    annotation.set_text(text_content)
    annotation.set_visible(True)
    plt.legend(loc = 'upper left')
    return line_king, line_normal, time_marker, water_level_dot, water_level_dot_2, annotation

ani = animation.FuncAnimation(fig_3, animate_both, frames = total_frames, interval = 50, blit = False, repeat = False)
ani.save('../../Downloads/tide_comparison_animation.mp4', writer = 'ffmpeg', fps = 60)
plt.tight_layout(rect = [0, 0.03, 1, 0.95])
plt.show()